In [4]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
%matplotlib inline
import warnings
warnings.filterwarnings('ignore')

In [5]:
ds=pd.read_csv('/home/aadil/ml_ws/MLProject-I/data/stud.csv')

In [6]:
ds.head()


,gender,race_ethnicity,parental_level_of_education,lunch,test_preparation_course,math_score,reading_score,writing_score
0,female,group B,bachelor's degree,standard,none,72,72,74
1,female,group C,some college,standard,completed,69,90,88
2,female,group B,master's degree,standard,none,90,95,93
3,male,group A,associate's degree,free/reduced,none,47,57,44
4,male,group C,some college,standard,none,76,78,75


In [7]:
ds.nunique()

gender                          2
race_ethnicity                  5
parental_level_of_education     6
lunch                           2
test_preparation_course         2
math_score                     81
reading_score                  72
writing_score                  77
dtype: int64

In [8]:
# Basic Import
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt 
import seaborn as sns
# Modelling
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor,AdaBoostRegressor
from sklearn.svm import SVR
from sklearn.linear_model import LinearRegression, Ridge,Lasso
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from sklearn.model_selection import RandomizedSearchCV
from catboost import CatBoostRegressor
from xgboost import XGBRegressor
import warnings

In [9]:
X=ds.drop(columns=['math_score'],axis=1)

In [10]:
X

,gender,race_ethnicity,parental_level_of_education,lunch,test_preparation_course,reading_score,writing_score
0,female,group B,bachelor's degree,standard,none,72,74
1,female,group C,some college,standard,completed,90,88
2,female,group B,master's degree,standard,none,95,93
3,male,group A,associate's degree,free/reduced,none,57,44
4,male,group C,some college,standard,none,78,75
...,...,...,...,...,...,...,...
995,female,group E,master's degree,standard,completed,99,95
996,male,group C,high school,free/reduced,none,55,55
997,female,group C,high school,free/reduced,completed,71,65
998,female,group D,some college,standard,completed,78,77


In [11]:
y=ds['math_score']

In [12]:
y

0      72
1      69
2      90
3      47
4      76
       ..
995    88
996    62
997    59
998    68
999    77
Name: math_score, Length: 1000, dtype: int64

In [13]:
num_features = X.select_dtypes(exclude="object").columns
cat_features = X.select_dtypes(include="object").columns

from sklearn.preprocessing import OneHotEncoder,StandardScaler
from sklearn.compose import ColumnTransformer

num_transformer = StandardScaler()
cat_transformer = OneHotEncoder()

preprocessor = ColumnTransformer([("OneHotEncoder",cat_transformer,cat_features),("StandardScaler",num_transformer,num_features)])


In [14]:
X = preprocessor.fit_transform(X)
print(num_features)
print((cat_features))

Index(['reading_score', 'writing_score'], dtype='object')
Index(['gender', 'race_ethnicity', 'parental_level_of_education', 'lunch',
       'test_preparation_course'],
      dtype='object')


In [15]:
X.shape

(1000, 19)

In [16]:
from sklearn.model_selection import train_test_split

X_test,X_train,y_test,y_train = train_test_split(X,y,random_state=42,train_size=0.2)


In [20]:
def evaluate_model(true,predicted):
    mae = mean_absolute_error(y_true=true,y_pred=predicted)
    mse = mean_squared_error(y_true=true,y_pred=predicted)
    rmse = np.sqrt(mse)
    r2_square = r2_score(y_true=true,y_pred=predicted)
    return mae,rmse,r2_square

In [17]:
models = {
    "AdaBoostRegressor":AdaBoostRegressor(),
    "LinearRegression":LinearRegression(),
    "Lasso":Lasso(),
    "DecisionTreeRegressor":DecisionTreeRegressor(),
    "Ridge":Ridge(),
    "XGBRegressor":XGBRegressor(),
    "KNeighborsRegressor":KNeighborsRegressor(),
    "RandomForestRegressor":RandomForestRegressor(),
    "SVR":SVR(),
    "CatBoostRegressor":CatBoostRegressor(verbose=False),
}

In [27]:
for model in models:
    models[model].fit(X_train,y_train)
    y_pred = models[model].predict(X_test)
    print(evaluate_model(y_test,y_pred))

(5.1261531915430325, 6.460988985347693, 0.8105971929728484)
(4.115468723455166, 5.150562726551254, 0.8796356787652492)
(4.999617822530243, 6.370763455916321, 0.8158501496767746)
(6.23, 7.77303029712351, 0.7258620689655173)
(4.115811372870263, 5.151138043958975, 0.879608787895093)
(5.220977783203125, 6.4423291509295, 0.8116896152496338)
(5.627999999999999, 7.205969747369191, 0.7644010889292197)
(4.798531666666666, 6.015309244521201, 0.8358260194772131)
(5.167331724690155, 7.153837953706653, 0.7677976521420427)
(4.614001332679719, 5.804552000931818, 0.8471287480420981)


In [1]:
def evaluate_model(X_train,y_train,X_test,y_test,models):
    report = {}
    for model in models:
        models[model].fit(X_train,y_train)
        y_pred = models[model].predict(X_test)
        r2_square = r2_score(y_true=y_test,y_pred=y_pred)
        report[model]=r2_square

    return report


In [23]:
model_report:dict = evaluate_model(X_train=X_train,y_train=y_train,X_test=X_test,y_test=y_test,models=models)
best_score = 0
for model in model_report:
    if(model_report[model]>best_score):
        best_model = model
        best_score=model_report[model]

best_model = models[best_model]
print(best_model)
print(r2_score(y_pred=best_model.predict(X_test),y_true=y_test))
print(model_report,best_score)

LinearRegression()
0.8796356787652492
{'AdaBoostRegressor': 0.8229270225787833, 'LinearRegression': 0.8796356787652492, 'Lasso': 0.8158501496767746, 'DecisionTreeRegressor': 0.7435344827586207, 'Ridge': 0.879608787895093, 'XGBRegressor': 0.8116896152496338, 'KNeighborsRegressor': 0.7644010889292197, 'RandomForestRegressor': 0.8348614923169994, 'SVR': 0.7677976521420427, 'CatBoostRegressor': 0.8471287480420981} 0.8796356787652492
